# 🚨 Triagem Rápida de Focos de Queimadas nas 29 UCs do Pará
### Núcleo de Geotecnologias (NGEO / IDEFLOR-Bio)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NGEO-IDEFLOR-Bio/cartografia-queimadas-ucs-pa/blob/main/notebooks/triagem_focos_ucs.ipynb)

Ferramenta de inteligência operacional de **alta velocidade** para varredura do **BDQueimadas / INPE** em todas as 29 Unidades de Conservação Estaduais do Pará.

⚡ **Zero Configuração de GEE:** Executa em segundos sem necessidade de autenticação no Earth Engine.

**Entregáveis em 1 Clique:**
1. 📊 **Quadro de Emergência com Níveis de Alerta** (🔴 Crítico, 🟠 Alerta, 🟡 Atenção, 🟢 Normal);
2. 📑 **Planilha Executiva (.xlsx)** com resumo das UCs e coordenadas completas de todos os focos (SIRGAS 2000);
3. 🌍 **Pacote KML para Google Earth / GPS** com as 29 UCs estilizadas e balões com síntese tática;
4. 🗺️ **Mapa Interativo (Folium)** para visualização instantânea de focos ativos e limites estaduais;
5. 🎯 **Direcionamento Direto** para o caderno de geração de Pranchas e Relatório Técnico (.docx).

In [ ]:
#@title 1. Instalação e Preparação do Ambiente { display-mode: "form" }
#@markdown *Clique no play para clonar o repositório e carregar os módulos oficiais.*

!pip install -q geopandas openpyxl folium

import os
import sys
from pathlib import Path

REPO_NAME = "cartografia-queimadas-ucs-pa"
if not Path(REPO_NAME).exists() and not Path("src").exists():
    print(f"[*] Clonando repositório oficial {REPO_NAME}...")
    !git clone https://github.com/NGEO-IDEFLOR-Bio/cartografia-queimadas-ucs-pa.git

if Path(REPO_NAME).exists():
    os.chdir(REPO_NAME)

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

print(f"[OK] Ambiente pronto para triagem! Diretório ativo: {os.getcwd()}")


In [ ]:
#@title 2. Parâmetros da Triagem de Emergência { display-mode: "form" }
#@markdown Selecione o intervalo temporal desejado para a varredura estadual:

JANELA_TEMPO = "Ultimos 2 dias (48 horas)" #@param ["Ultimas 24 horas (1 dia)", "Ultimos 2 dias (48 horas)", "Ultimos 7 dias (1 semana)", "Ultimos 15 dias", "Ultimos 30 dias (1 mes)", "Personalizado (datas abaixo)"]
DATA_INICIAL_CUSTOM = "" #@param {type:"string"}
DATA_FINAL_CUSTOM = "" #@param {type:"string"}

from datetime import date, timedelta

mapa_dias = {
    "Ultimas 24 horas (1 dia)": 1,
    "Ultimos 2 dias (48 horas)": 2,
    "Ultimos 7 dias (1 semana)": 7,
    "Ultimos 15 dias": 15,
    "Ultimos 30 dias (1 mes)": 30
}

if JANELA_TEMPO == "Personalizado (datas abaixo)" and DATA_INICIAL_CUSTOM and DATA_FINAL_CUSTOM:
    data_inicio_proc = DATA_INICIAL_CUSTOM.strip()
    data_fim_proc = DATA_FINAL_CUSTOM.strip()
    dias_proc = None
    print(f"[+] Modo Personalizado selecionado: {data_inicio_proc} até {data_fim_proc}")
else:
    dias_proc = mapa_dias.get(JANELA_TEMPO, 2)
    data_fim_proc = date.today().strftime("%Y-%m-%d")
    data_inicio_proc = (date.today() - timedelta(days=dias_proc)).strftime("%Y-%m-%d")
    print(f"[+] Janela selecionada ({JANELA_TEMPO}): {data_inicio_proc} até {data_fim_proc}")


In [ ]:
#@title 3. Executar Varredura e Exibir Quadro de Emergência { display-mode: "form" }
#@markdown *Cruza todos os focos do Pará com as 29 UCs e ranqueia as ocorrências.*

from IPython.display import display, HTML
from src.triagem import executar_triagem_focos_ucs

resultado_triagem = executar_triagem_focos_ucs(
    dias=dias_proc or 2,
    data_inicio=data_inicio_proc,
    data_fim=data_fim_proc,
    out_dir="outputs/triagem"
)

df_ranking = resultado_triagem["df_ranking"]

# Exibição rica do ranking de UCs no Colab
cols_view = ["Alerta", "Unidade de Conservação", "Total Focos", "Focos Recentes (< 48h)", "Focos Anteriores", "FRP Máximo (MW)", "Municípios Atingidos"]
df_show = df_ranking[cols_view].copy()

def highlight_alert(row):
    alerta = row["Alerta"]
    if "CRÍTICO" in alerta:
        return ["background-color: #FFCDD2; font-weight: bold; color: #B71C1C;"] * len(row)
    elif "ALERTA" in alerta:
        return ["background-color: #FFE0B2; font-weight: bold; color: #E65100;"] * len(row)
    elif "ATENÇÃO" in alerta:
        return ["background-color: #FFF9C4; color: #F57F17;"] * len(row)
    else:
        return ["background-color: #E8F5E9; color: #2E7D32;"] * len(row)

styled_table = df_show.style.apply(highlight_alert, axis=1).format({
    "Total Focos": "{:,.0f}",
    "Focos Recentes (< 48h)": "{:,.0f}",
    "Focos Anteriores": "{:,.0f}",
    "FRP Máximo (MW)": "{:.1f}"
})

display(styled_table)


In [ ]:
#@title 4. Mapa Interativo Estadual (Folium) { display-mode: "form" }
#@markdown *Visualize a distribuição espacial do fogo nas 29 UCs e clique nos polígonos para detalhes.*

mapa = resultado_triagem.get("mapa_folium")
if mapa:
    display(mapa)
else:
    print("Mapa interativo não disponível.")


In [ ]:
#@title 5. Baixar Pacote de Triagem (ZIP) e Selecionar UC { display-mode: "form" }
#@markdown *Cria pacote .zip com a Planilha (.xlsx), o KML Estadual e o Mapa HTML para download imediato.*

import zipfile
from pathlib import Path
from IPython.display import display, HTML

periodo_str = f"{resultado_triagem['periodo'][0]}_a_{resultado_triagem['periodo'][1]}"
zip_name = f"Triagem_Focos_UCs_{periodo_str}.zip"
zip_path = Path("outputs/triagem") / zip_name

arquivos_incluidos = []
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for chave in ["planilha_path", "kml_path", "html_path"]:
        p = resultado_triagem.get(chave)
        if p and Path(p).exists():
            zipf.write(p, arcname=Path(p).name)
            arquivos_incluidos.append(Path(p).name)

print(f"[OK] Pacote ZIP gerado: {zip_name} ({zip_path.stat().st_size / 1024:.1f} KB)")
print("Arquivos incluídos:")
for arq in arquivos_incluidos:
    print(f"  • {arq}")

# Download automático no Colab
try:
    from google.colab import files
    print("\n[*] Iniciando download direto no seu navegador...")
    files.download(str(zip_path))
except ImportError:
    print(f"\n[!] Ambiente local detectado. O arquivo está salvo em: {zip_path.resolve()}")

# Guia para o Gerador Cartográfico
ucs_focos = resultado_triagem.get("ucs_com_focos", [])
if ucs_focos:
    top_uc = ucs_focos[0]
    print("\n" + "=" * 70)
    print("🎯 PRÓXIMO PASSO RECOMENDADO:")
    print(f"A UC mais crítica identificada é: '{top_uc}'")
    print("Para gerar as 3 Pranchas A4 (300 DPI), o dNBR e o Relatório Word Oficial (.docx):")
    print("1. Abra o caderno do Gerador Cartográfico (colab_runner.ipynb);
    print(f"2. Cole o nome: {top_uc} no campo 'NOME_UC' e execute!")
    print("=" * 70)
